In [8]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [9]:
with open('input.txt','r',encoding='utf-8') as f:
    text = f.read()

print(len(text))
print(text[:200])

1115393
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [10]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

In [11]:
print(encode("hello"))

[46, 43, 50, 50, 53]


CONVERT DATASET TO TENSOR

In [12]:
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

BATCH GENERATOR

Transformers train in batches.

In [13]:
block_size = 64
batch_size = 32

def get_batch(split):

    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x.to(device), y.to(device)

POSITIONAL EMBEDDING

Transformers must know word positions

In [14]:
class PositionalEmbedding(nn.Module):

    def __init__(self, block_size, embed_dim):
        super().__init__()
        self.pos_embedding = nn.Embedding(block_size, embed_dim)

    def forward(self, x):

        B,T = x.shape
        positions = torch.arange(T, device=device)

        return self.pos_embedding(positions)

SELF ATTENTION HEAD

In [15]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(embed_dim, head_size, bias=False)
        self.query = nn.Linear(embed_dim, head_size, bias=False)
        self.value = nn.Linear(embed_dim, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):

        B,T,C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2,-1) * (C ** -0.5)

        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))

        wei = F.softmax(wei, dim=-1)

        v = self.value(x)

        out = wei @ v

        return out

MULTI HEAD ATTENTION

In [16]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        out = torch.cat([h(x) for h in self.heads], dim=-1)

        return self.proj(out)

FEED FORWARD NETWORK

Small neural network inside transformer

In [17]:
class FeedForward(nn.Module):

    def __init__(self, embed_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4*embed_dim),
            nn.ReLU(),
            nn.Linear(4*embed_dim, embed_dim)
        )

    def forward(self, x):
        return self.net(x)

TRANSFORMER BLOCK

Combine attention + feedforward.

In [18]:
class Block(nn.Module):

    def __init__(self, embed_dim, num_heads):
        super().__init__()

        head_size = embed_dim // num_heads

        self.sa = MultiHeadAttention(num_heads, head_size)

        self.ff = FeedForward(embed_dim)

        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x):

        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))

        return x

FULL TRANSFORMER MODEL

Now combine everything.

In [19]:
embed_dim = 128
num_heads = 4
num_layers = 4

class GPT(nn.Module):

    def __init__(self):

        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, embed_dim)

        self.position_embedding = nn.Embedding(block_size, embed_dim)

        self.blocks = nn.Sequential(
            *[Block(embed_dim, num_heads) for _ in range(num_layers)]
        )

        self.ln = nn.LayerNorm(embed_dim)

        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding(idx)

        pos_emb = self.position_embedding(torch.arange(T, device=device))

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln(x)

        logits = self.fc(x)

        if targets is None:
            loss = None
        else:
            logits = logits.view(-1, vocab_size)
            targets = targets.view(-1)

            loss = F.cross_entropy(logits, targets)

        return logits, loss


    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx

CREATE MODEL

In [20]:
model = GPT().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

TRAINING LOOP

In [21]:
max_iters = 5000

for iter in range(max_iters):

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if iter % 500 == 0:
        print(iter, loss.item())

0 4.235130310058594
500 2.2747585773468018
1000 1.9898254871368408
1500 1.9609761238098145
2000 1.8511261940002441
2500 1.8429440259933472
3000 1.71946120262146
3500 1.7252501249313354
4000 1.5845422744750977
4500 1.5489299297332764


In [22]:
context = torch.zeros((1,1), dtype=torch.long, device=device)

generated = model.generate(context, max_new_tokens=500)

print(decode(generated[0].tolist()))


The faultler'd Richards dost his quarter'
Condrain hath brail meet in deheer us monest.

Doisford Wary!

So Musber:
Seeten, there good's to ripe, conce, my lord!
Desire besore; Destry Angerous,
I but be trituse threal noth off
Wlood, to would corn'd in gacnoreary, hight a canse?

MERCUTIO:
And love, Be dare deatirance,--


PeTRICHER:
Brown judher's namys majest of live outherfore.
Vorabruath, Warwere kins,
And, foot foot and iptisirat: He know man you,
Didsting for a confedyiance, my crown?
Shal


Tokenization

↓

Embeddings

↓

Transformer Blocks

↓

Language Model Head

↓

Next Token Prediction

In [23]:
import matplotlib.pyplot as plt
%matplotlib inline
def visualize_attention(attention_matrix):

    plt.figure(figsize=(6,6))
    plt.imshow(attention_matrix, cmap="viridis")
    plt.colorbar()
    plt.title("Attention Weights")
    plt.xlabel("Key Positions")
    plt.ylabel("Query Positions")
    plt.show()

In [24]:
import gradio as gr
import torch

model.eval()

def generate_text(prompt):

    tokens = encode(prompt)

    context = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

    context = context[:, -block_size:]

    output = model.generate(context, max_new_tokens=200)

    return decode(output[0].tolist())

demo = gr.Interface(
    fn=generate_text,
    inputs="text",
    outputs="text",
    title="Mini GPT Text Generator"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
